In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/kredit_sumatera.csv")

print("Tingkat gagal bayar:", f"{df.gagal_bayar.mean():.2%}")
print(pd.crosstab(df.periode.str[:4], df.segmen, normalize="index").round(3))

Tingkat gagal bayar: 8.30%
segmen   nelayan  pedagang_musiman  pegawai_tetap  petani  wiraswasta
periode                                                              
2022       0.067             0.139          0.421    0.13       0.242
2023       0.067             0.142          0.419    0.13       0.241
2024       0.099             0.171          0.339    0.16       0.231


In [2]:
hilang = df.isna().mean().sort_values(ascending=False)
print(hilang[hilang > 0].round(4))
print()
print(df.groupby("segmen")["rata_saldo_90h"].apply(lambda s: s.isna().mean()).round(3))
print()
print(df.groupby("gagal_bayar")["status_restrukturisasi"].apply(lambda s: s.notna().mean()))

status_restrukturisasi    0.9479
lama_usaha_tahun          0.0294
rata_saldo_90h            0.0231
dtype: float64

segmen
nelayan             0.108
pedagang_musiman    0.000
pegawai_tetap       0.000
petani              0.105
wiraswasta          0.000
Name: rata_saldo_90h, dtype: float64

gagal_bayar
0    0.000000
1    0.627509
Name: status_restrukturisasi, dtype: float64


In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df["rasio_angsuran"] = df.angsuran_bulanan / df.penghasilan_bulanan

num = ["usia", "tanggungan", "lama_usaha_tahun", "penghasilan_bulanan",
       "ragam_penghasilan", "plafon_diajukan", "tenor_bulan", "bunga_tahunan",
       "rasio_angsuran", "rata_saldo_90h", "jumlah_tunggakan_12b"]
X = df[num].join(pd.get_dummies(df[["segmen", "wilayah", "kanal"]]))
y = df.gagal_bayar

latih = df.periode <= "2023-12"
uji = df.periode > "2024-06"

m = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                   max_iter=250, random_state=42)
m.fit(X[latih], y[latih])
skor = m.predict_proba(X[uji])[:, 1]
print("AUC out-of-time:", round(roc_auc_score(y[uji], skor), 4))

AUC out-of-time: 0.7744


In [4]:
from sklearn.model_selection import train_test_split

Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m2 = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                    max_iter=250, random_state=42).fit(Xa, ya)
print("AUC pembagian acak:", round(roc_auc_score(yb, m2.predict_proba(Xb)[:, 1]), 4))

AUC pembagian acak: 0.7631


In [5]:
import numpy as np

hasil = []
for seed in range(5):
    a, b, ta, tb = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    mm = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                        max_iter=250, random_state=42).fit(a, ta)
    hasil.append(roc_auc_score(tb, mm.predict_proba(b)[:, 1]))

print("AUC acak per seed:", np.round(hasil, 4))
print("Rata-rata:", round(np.mean(hasil), 4))
print("Model acak diuji out-of-time:", round(roc_auc_score(y[uji], m2.predict_proba(X[uji])[:, 1]), 4))

AUC acak per seed: [0.7771 0.7994 0.7785 0.7749 0.7798]
Rata-rata: 0.782
Model acak diuji out-of-time: 0.85
